# Choosing `lam` for each potential set

Reads the offline `lam` sweep written by `launch/resolve_lamsweep.sh`
(`saved_results/theta_reg_lamsweep/<config>/lam<lam>_ridge<ridge>.pt`) for the
`lamtune_*` tuning runs, and measures for every set and `lam`:

* **bias** — residual `R = Theta_reg(lam) - Theta_reg(0)`. `lam = 0` is the
  unsmoothed per-node solve of the same system (same solver, no time coupling),
  so it is the right reference: `theta_t` is *not*, because the per-step solves
  behind it carry the 0.01 relative ridge (`--regularization`), which would show
  up as smooth, signal-shaped residual even at `lam -> 0`.
  While `lam` only removes noise, `R` is (high-passed) noise and its block means
  over `w` nodes shrink like `1/sqrt(w)`: the block z-score
  `z = mean_block(R) / (std(R)/sqrt(w))` has `E[z^2] ~ 1` (or below).
  Once `lam` starts smoothing away real structure, `R` gains a slow component
  that does not average out, and `E[z^2]` grows well above 1.
* **variance** — across-seed variance of `Theta_reg(lam)` relative to that of
  `Theta_reg(0)` (the variance gain from smoothing).
* **final node** — across-seed spread of `Theta_reg[-1]` (the `theta` used
  downstream) relative to `lam = 0`. (Its bias is covered by the last window's
  `E[z^2]`: at a single node the `lam = 0` reference is too noisy to measure a
  shift against.)

All three are reported per time window, since `lam` acts through `lam/dt^2` and
the Cos schedule's steps shrink near `t = 1`.

**Rule of thumb for the choice:** the largest `lam` with `E[z^2] <= Z2_MAX = 1`
in every window (at the block size `W_DECIDE`), then check the variance gain.
Calibrated on one synthetic test with a known `theta(t)` (smooth sinusoids plus
white per-step noise): `Z2_MAX = 1` landed on the true RMS-error optimum,
`Z2_MAX = 2` overshot it by one half-decade (1.6x the optimal error). This is a default, not a verdict — look at
the heatmap and the trajectories before fixing a value.

Memory: one `Theta_reg` is `n_nodes x r x 8` bytes (~93 MB for the turbulence
runs), so everything is streamed one (seed, lam) file at a time.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# ── parameters ────────────────────────────────────────────────────────────
# ROOT: the folder holding saved_results/ (this notebook's folder, turbulence/).
ROOT = Path(os.environ.get('LAMTUNE_ROOT', Path.cwd()))
LAMSWEEP_DIR = ROOT / 'saved_results' / 'theta_reg_lamsweep'

SETS = [                        # run labels, as set by the launch/lamtune_*.sh scripts
    'lamtune_full',
    'lamtune_noL2lowpass',
    'lamtune_noScalarMorlet',
    'lamtune_noL2lowpass_noScalarMorlet',
]
RIDGE = 1e-6                    # the RIDGE resolve_lamsweep.sh solved with
WINDOWS = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 1.0]   # t-windows for all diagnostics
BLOCKS = [10, 50, 200]          # block sizes (nodes) for the residual z^2 test
W_DECIDE = 50                   # block size the automatic suggestion uses
Z2_MAX = 1.0                    # accepted E[z^2]: slow residual no larger than noise

print('ROOT        :', ROOT)
print('LAMSWEEP_DIR:', LAMSWEEP_DIR, '(exists)' if LAMSWEEP_DIR.is_dir() else '(MISSING)')

## Discover runs and `lam` values

In [ ]:
NAME_RE = re.compile(r'_seed_(\d+)_terms([0-9a-f]{8})_(.+)_(\d{8}_\d{4})$')
LAM_RE = re.compile(r'^lam(.+)_ridge(.+)\.pt$')

def parse_config(config):
    m = NAME_RE.search(config)
    return None if m is None else dict(seed=int(m[1]), terms=m[2], label=m[3], timestamp=m[4])

runs = {s: [] for s in SETS}            # label -> [config, ...]
for d in sorted(p for p in LAMSWEEP_DIR.iterdir() if p.is_dir()):
    info = parse_config(d.name)
    if info is not None and info['label'] in runs:
        runs[info['label']].append(d.name)

def lams_of(config):
    out = set()
    for f in (LAMSWEEP_DIR / config).glob('lam*_ridge*.pt'):
        m = LAM_RE.match(f.name)
        if m and float(m[2]) == RIDGE:
            out.add(float(m[1]))
    return out

lams = {}                                # label -> sorted lam values present for EVERY seed
for label, configs in runs.items():
    if configs:
        common = set.intersection(*(lams_of(c) for c in configs))
        lams[label] = sorted(common)

for label in SETS:
    cs = runs[label]
    seeds = sorted(parse_config(c)['seed'] for c in cs)
    print(f"{label:38s} {len(cs)} seeds {seeds}  lams: {lams.get(label, [])}")
    if cs and 0.0 not in lams[label]:
        print('   !! no lam=0 reference solved for every seed -- add 0 to LAM_LIST in resolve_lamsweep.sh')

## Diagnostics (streamed)

In [ ]:
def load_sweep(config, lam):
    d = torch.load(LAMSWEEP_DIR / config / f'lam{lam:g}_ridge{RIDGE:g}.pt',
                   map_location='cpu', weights_only=False)
    return d['Theta_reg'].double().numpy(), d['t_reg'].double().numpy(), d['residual']

def window_masks(t_reg):
    return [(lo, hi, (t_reg >= lo) & (t_reg < hi if hi < 1.0 else t_reg <= hi))
            for lo, hi in zip(WINDOWS[:-1], WINDOWS[1:])]

def block_z2(R, w):
    # E[z^2] of block means of the residual R (n, r), pooled over blocks and coefficients
    n = (len(R) // w) * w
    if n < 2 * w:
        return np.nan
    sd = R.std(0, ddof=1)
    ok = sd > 0
    if not ok.any():
        return np.nan
    z = R[:n, ok].reshape(-1, w, ok.sum()).mean(1) / (sd[ok] / np.sqrt(w))
    return float(np.mean(z ** 2))

def analyse_set(label):
    configs, lam_list = runs[label], lams[label]
    ref = {c: load_sweep(c, 0.0) for c in configs}                # (Theta, t_reg, residual)
    t_reg = next(iter(ref.values()))[1]
    for c, (_, tc, _) in ref.items():
        assert len(tc) == len(t_reg) and np.allclose(tc, t_reg), f'{c}: t_reg differs across seeds'
    masks = window_masks(t_reg)
    ref_stack = np.stack([ref[c][0] for c in configs])            # (S, n, r)
    var_ref = ref_stack.var(0, ddof=1) if len(configs) > 1 else None
    fin_ref = ref_stack[:, -1, :]                                 # (S, r)

    z2 = np.full((len(lam_list), len(masks), len(BLOCKS)), np.nan)   # seed-averaged
    vgain = np.full((len(lam_list), len(masks)), np.nan)
    final_std_ratio = np.full(len(lam_list), np.nan)
    max_resid = np.zeros(len(lam_list))

    for i, lam in enumerate(lam_list):
        s1 = s2 = None
        fin = []
        z2_seed = []
        for c in configs:
            Th, tc, res = load_sweep(c, lam)
            max_resid[i] = max(max_resid[i], res)
            if lam > 0:                                           # lam = 0 is the reference itself
                R = Th - ref[c][0]
                z2_seed.append([[block_z2(R[m], w) for w in BLOCKS] for _, _, m in masks])
                del R
            s1 = Th.copy() if s1 is None else s1 + Th
            s2 = Th ** 2 if s2 is None else s2 + Th ** 2
            fin.append(Th[-1])
            del Th
        if z2_seed:
            with warnings.catch_warnings():   # all-NaN where a window is shorter than 2 blocks
                warnings.simplefilter('ignore', RuntimeWarning)
                z2[i] = np.nanmean(np.array(z2_seed, dtype=float), axis=0)
        S = len(configs)
        if S > 1:
            var = (s2 - s1 ** 2 / S) / (S - 1)
            for j, (_, _, m) in enumerate(masks):
                vr = var[m] / np.where(var_ref[m] > 0, var_ref[m], np.nan)
                vgain[i, j] = np.nanmedian(vr)
            fin = np.stack(fin)
            sd_ref = fin_ref.std(0, ddof=1)
            final_std_ratio[i] = np.nanmedian(fin.std(0, ddof=1) / np.where(sd_ref > 0, sd_ref, np.nan))
    return dict(lams=lam_list, masks=[(lo, hi, int(m.sum())) for lo, hi, m in masks],
                z2=z2, vgain=vgain, final_std_ratio=final_std_ratio,
                max_resid=max_resid, n_seeds=len(configs), t_reg=t_reg)

results = {}
for label in SETS:
    if runs[label] and 0.0 in lams.get(label, []):
        results[label] = analyse_set(label)
        print(f'{label}: done ({results[label]["n_seeds"]} seeds, {len(results[label]["lams"])} lams)')

## Summary and suggested `lam` per set

In [ ]:
def suggest(res):
    b = BLOCKS.index(W_DECIDE)
    ok = [lam for i, lam in enumerate(res['lams'])
          if lam > 0 and np.all(np.nan_to_num(res['z2'][i, :, b], nan=0.0) <= Z2_MAX)]
    return max(ok) if ok else None

suggestions = {}
for label, res in results.items():
    b = BLOCKS.index(W_DECIDE)
    suggestions[label] = suggest(res)
    print(f"\n=== {label}  ({res['n_seeds']} seeds)   suggested lam = {suggestions[label]}")
    print(f"max solve residual over all (seed, lam): {res['max_resid'].max():.1e}")
    hdr = ' '.join(f'[{lo:.2f},{hi:.2f})' for lo, hi, _ in res['masks'])
    print(f"{'lam':>8} | E[z^2] w={W_DECIDE} per window: {hdr} | var gain (median, worst window) | final std ratio")
    for i, lam in enumerate(res['lams']):
        zs = ' '.join(f'{v:13.2f}' for v in res['z2'][i, :, b])
        vg = np.nanmax(res['vgain'][i]) if np.isfinite(res['vgain'][i]).any() else np.nan
        print(f"{lam:8.0e} | {zs} | {vg:8.3f} | {res['final_std_ratio'][i]:8.3f}")

In [ ]:
%matplotlib inline
for label, res in results.items():
    L = [lam for lam in res['lams'] if lam > 0]
    idx = [res['lams'].index(lam) for lam in L]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
    b = BLOCKS.index(W_DECIDE)
    im = axes[0].imshow(np.log10(res['z2'][idx, :, b].T), aspect='auto', cmap='magma',
                        vmin=-1, vmax=2, origin='lower')
    axes[0].set_xticks(range(len(L))); axes[0].set_xticklabels([f'{l:.0e}' for l in L], rotation=60)
    axes[0].set_yticks(range(len(res['masks'])))
    axes[0].set_yticklabels([f'[{lo:.2f},{hi:.2f})' for lo, hi, _ in res['masks']])
    axes[0].set_title(f'log10 E[z^2] (w={W_DECIDE}); bias where >> 0')
    plt.colorbar(im, ax=axes[0])
    for j, (lo, hi, _) in enumerate(res['masks']):
        axes[1].loglog(L, res['vgain'][idx, j], 'o-', ms=3, label=f'[{lo:.2f},{hi:.2f})')
    axes[1].set_xlabel('lam'); axes[1].set_title('across-seed variance / variance at lam=0')
    axes[1].legend(fontsize=7)
    axes[2].loglog(L, res['final_std_ratio'][idx], 'o-')
    axes[2].set_xlabel('lam'); axes[2].set_title('final node: across-seed std / std at lam=0')
    if suggestions[label] is not None:
        for ax in axes[1:]:
            ax.axvline(suggestions[label], color='tab:green', lw=1, ls=':')
    fig.suptitle(label); plt.tight_layout(); plt.show()

## Trajectories for one seed and coefficient

Sanity check by eye: the suggested `lam` should follow the unsmoothed curve's trend without flattening real structure.

In [ ]:
%matplotlib inline
LABEL = next(iter(results), None)       # pick a set
COEF = 0                                 # coefficient index
if LABEL is not None:
    config = runs[LABEL][0]
    th0, t_reg, _ = load_sweep(config, 0.0)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t_reg, th0[:, COEF], color='0.75', lw=0.5, label='lam = 0 (unsmoothed)')
    show = [l for l in lams[LABEL] if l > 0][::3]
    if suggestions.get(LABEL) and suggestions[LABEL] not in show:
        show.append(suggestions[LABEL])
    for lam in sorted(show):
        th, _, _ = load_sweep(config, lam)
        ax.plot(t_reg, th[:, COEF], lw=1.8 if lam == suggestions.get(LABEL) else 1,
                label=f'lam = {lam:g}' + ('  (suggested)' if lam == suggestions.get(LABEL) else ''))
    lo, hi = np.percentile(th0[:, COEF], [1, 99])
    ax.set_ylim(lo - 0.2 * (hi - lo), hi + 0.2 * (hi - lo))
    ax.set_xlabel('SDE time t'); ax.set_ylabel(f'Theta_reg[:, {COEF}]')
    ax.set_title(f'{LABEL}, seed {parse_config(config)["seed"]}'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()